In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

import numpy as np

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date
import matplotlib.pyplot as plt

In [7]:
def get_ipos() -> pd.DataFrame:
    """
    Fetch IPO data for the given year from stockanalysis.com.
    """
    url = f"https://www.iposcoop.com/2025-pricings/"
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Wrap HTML text in StringIO to avoid deprecation warning
        # "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
        html_io = StringIO(response.text)
        tables = pd.read_html(html_io)

        if not tables:
            raise ValueError(f"No tables found")

        return tables[0]

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except ValueError as ve:
        print(f"Data error: {ve}")
    except Exception as ex:
        print(f"Unexpected error: {ex}")

    return pd.DataFrame()

In [8]:
raw_df = get_ipos()

In [9]:
df = raw_df.copy()

In [11]:
df["Offer Date"] = pd.to_datetime(df["Offer Date"], errors="coerce")

In [14]:
offer_date_filter = (df["Offer Date"] < pd.to_datetime("2025-09-01"))

In [15]:
filtered_df = df[offer_date_filter].copy()

In [16]:
filtered_df2 = filtered_df.copy()

In [17]:
filtered_df2 = filtered_df2.rename(columns={"Return": "Return (Percentage)"})

In [18]:
filtered_df2["Return (Percentage)"] = pd.to_numeric(filtered_df2["Return (Percentage)"].str.replace("%", "", regex=False),
    errors="coerce")

In [19]:
filtered_df2.columns

Index(['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)',
       'Offer Price', '1st Day Close', 'Current Price', 'Return (Percentage)',
       'SCOOP Rating'],
      dtype='object')

In [20]:
filtered_df2["Return (Percentage)"].max()

675.1

In [21]:
filter_returns = ~(filtered_df2["Return (Percentage)"] == 0)

In [22]:
df2 = filtered_df2.copy()

In [23]:
df2 = filtered_df2[filter_returns]

In [24]:
df2

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return (Percentage),SCOOP Rating
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00,S/O
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00,S/O
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75,S/O
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,$4.00,$4.02,$0.37,-90.75,S/O
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,$10.00,$10.12,$10.12,1.20,S/O
...,...,...,...,...,...,...,...,...,...,...
226,Mint Inc. Ltd.,MIMI,Other,2025-01-10,1.8,$4.00,$4.11,$5.30,32.50,S/O
227,3 E Network Technology Group Limited,MASK,Technology,2025-01-08,1.3,$4.00,$3.25,$2.75,-31.25,S/O
228,"Cantor Equity Partners I, Inc. (Stock-Only SPAC)",CEPO,Blank Check,2025-01-07,20.0,$10.00,$10.05,$10.10,1.00,S/O
229,Zhengye Biotechnology Holding Limited,ZYBT,Health Care,2025-01-07,1.5,$4.00,$4.86,$4.22,5.50,S/O


In [26]:
df3 = list(df2["Symbol"])

In [29]:
stocks = []
for ticker_name in df3:
	ticker_obj = yf.Ticker(ticker_name)
	stocks.append(ticker_obj)

In [77]:
history_list = []

for item in stocks:
    item_history = item.history(
        period="1y",
        interval="1d"
    )

    if not item_history.empty:
        item_history = item_history.reset_index()
        item_history["Ticker"] = item.ticker

        history_list.append(item_history)

concat_df = pd.concat(
    history_list,
    ignore_index=True
)

$MJID: No data found, symbol may be delisted
$EMPG: No data found, symbol may be delisted
$CAEP: No data found, symbol may be delisted
$PTNM: No data found, symbol may be delisted
$AHL: No data found, symbol may be delisted
$CEPT: No data found, symbol may be delisted
$SDM: No data found, symbol may be delisted
$TBH: No data found, symbol may be delisted
$WGRX: No data found, symbol may be delisted
$AGH: No data found, symbol may be delisted
$EPWK: No data found, symbol may be delisted
$MTSR: No data found, symbol may be delisted
$SKBL: No data found, symbol may be delisted
$MCTR: No data found, symbol may be delisted


In [81]:
main_df = concat_df.copy()

In [123]:
main_df

,Date,Open,High,Low,Close,Volume,Ticker,volatility,growth_252d,Sharpe
0,2025-09-29 00:00:00-04:00,426.50,435.000000,405.000,435.00,17242,PMI,NaN,NaN,NaN
1,2025-09-30 00:00:00-04:00,428.50,441.899994,422.750,425.00,3446,PMI,NaN,NaN,NaN
2,2025-10-01 00:00:00-04:00,425.50,447.500000,425.500,442.00,2436,PMI,NaN,NaN,NaN
3,2025-10-02 00:00:00-04:00,462.00,462.000000,432.500,457.50,43136,PMI,NaN,NaN,NaN
4,2025-10-03 00:00:00-04:00,453.50,481.500000,407.500,481.50,51602,PMI,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
32889,2026-09-22 00:00:00-04:00,2.65,2.950000,2.650,2.80,40400,INLF,12.137704,2.074074,0.166759
32890,2026-09-23 00:00:00-04:00,2.86,2.910000,2.834,2.91,16300,INLF,12.083139,2.273438,0.184012
32891,2026-09-24 00:00:00-04:00,2.90,3.000000,2.890,2.95,9533500,INLF,11.880556,2.153285,0.177036
32892,2026-09-25 00:00:00-04:00,6.25,6.870000,4.570,5.10,49411900,INLF,12.155550,3.893130,0.316163


In [84]:
main_df.dtypes

Date             datetime64[ns, America/New_York]
Open                                      float64
High                                      float64
Low                                       float64
Close                                     float64
Volume                                      int64
Dividends                                 float64
Stock Splits                              float64
Ticker                                     object
Capital Gains                             float64
dtype: object

In [85]:
main_df['volatility'] = main_df['Close'].rolling(30).std() * np.sqrt(252)

In [87]:
main_df.drop(
    columns=["Dividends", "Stock Splits", "Capital Gains"],
    inplace=True
)

In [88]:
main_df

,Date,Open,High,Low,Close,Volume,Ticker,volatility
0,2025-09-29 00:00:00-04:00,426.50,435.000000,405.000,435.00,17242,PMI,NaN
1,2025-09-30 00:00:00-04:00,428.50,441.899994,422.750,425.00,3446,PMI,NaN
2,2025-10-01 00:00:00-04:00,425.50,447.500000,425.500,442.00,2436,PMI,NaN
3,2025-10-02 00:00:00-04:00,462.00,462.000000,432.500,457.50,43136,PMI,NaN
4,2025-10-03 00:00:00-04:00,453.50,481.500000,407.500,481.50,51602,PMI,NaN
...,...,...,...,...,...,...,...,...
32889,2026-09-22 00:00:00-04:00,2.65,2.950000,2.650,2.80,40400,INLF,12.137704
32890,2026-09-23 00:00:00-04:00,2.86,2.910000,2.834,2.91,16300,INLF,12.083139
32891,2026-09-24 00:00:00-04:00,2.90,3.000000,2.890,2.95,9533500,INLF,11.880556
32892,2026-09-25 00:00:00-04:00,6.25,6.870000,4.570,5.10,49411900,INLF,12.155550


In [91]:
main_df["growth_252d"] = (main_df["Close"] / main_df["Close"].shift(252))

In [92]:
main_df

,Date,Open,High,Low,Close,Volume,Ticker,volatility,growth_252d
0,2025-09-29 00:00:00-04:00,426.50,435.000000,405.000,435.00,17242,PMI,NaN,NaN
1,2025-09-30 00:00:00-04:00,428.50,441.899994,422.750,425.00,3446,PMI,NaN,NaN
2,2025-10-01 00:00:00-04:00,425.50,447.500000,425.500,442.00,2436,PMI,NaN,NaN
3,2025-10-02 00:00:00-04:00,462.00,462.000000,432.500,457.50,43136,PMI,NaN,NaN
4,2025-10-03 00:00:00-04:00,453.50,481.500000,407.500,481.50,51602,PMI,NaN,NaN
...,...,...,...,...,...,...,...,...,...
32889,2026-09-22 00:00:00-04:00,2.65,2.950000,2.650,2.80,40400,INLF,12.137704,2.074074
32890,2026-09-23 00:00:00-04:00,2.86,2.910000,2.834,2.91,16300,INLF,12.083139,2.273438
32891,2026-09-24 00:00:00-04:00,2.90,3.000000,2.890,2.95,9533500,INLF,11.880556,2.153285
32892,2026-09-25 00:00:00-04:00,6.25,6.870000,4.570,5.10,49411900,INLF,12.155550,3.893130


In [93]:
main_df['Sharpe'] = (main_df['growth_252d'] - 0.05) / main_df['volatility']

In [94]:
main_df

,Date,Open,High,Low,Close,Volume,Ticker,volatility,growth_252d,Sharpe
0,2025-09-29 00:00:00-04:00,426.50,435.000000,405.000,435.00,17242,PMI,NaN,NaN,NaN
1,2025-09-30 00:00:00-04:00,428.50,441.899994,422.750,425.00,3446,PMI,NaN,NaN,NaN
2,2025-10-01 00:00:00-04:00,425.50,447.500000,425.500,442.00,2436,PMI,NaN,NaN,NaN
3,2025-10-02 00:00:00-04:00,462.00,462.000000,432.500,457.50,43136,PMI,NaN,NaN,NaN
4,2025-10-03 00:00:00-04:00,453.50,481.500000,407.500,481.50,51602,PMI,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
32889,2026-09-22 00:00:00-04:00,2.65,2.950000,2.650,2.80,40400,INLF,12.137704,2.074074,0.166759
32890,2026-09-23 00:00:00-04:00,2.86,2.910000,2.834,2.91,16300,INLF,12.083139,2.273438,0.184012
32891,2026-09-24 00:00:00-04:00,2.90,3.000000,2.890,2.95,9533500,INLF,11.880556,2.153285,0.177036
32892,2026-09-25 00:00:00-04:00,6.25,6.870000,4.570,5.10,49411900,INLF,12.155550,3.893130,0.316163


In [117]:
date_filt = main_df["Date"] == ("2026-09-11")

In [118]:
stocks_main_filtered = main_df[date_filt]

In [121]:
stocks_main_filtered.describe()

,Open,High,Low,Close,Volume,volatility,growth_252d,Sharpe
count,132.000000,132.000000,132.000000,132.000000,1.320000e+02,132.000000,131.000000,131.000000
mean,12.831901,13.181377,12.510782,12.800431,1.248240e+06,28.116148,7.254812,inf
std,19.213619,19.564220,18.868847,19.135136,3.184362e+06,66.894326,22.790693,NaN
min,0.116000,0.129000,0.115000,0.125000,0.000000e+00,0.000000,0.012860,-0.066043
25%,2.137500,2.257500,2.075500,2.098750,2.297500e+04,2.338723,0.220758,0.019977
50%,4.505000,4.790000,4.475000,4.565000,1.708500e+05,8.164477,0.996835,0.096236
75%,17.397501,17.471250,16.732501,17.224999,9.976250e+05,23.942749,3.465440,0.456654
max,120.500000,120.500000,120.425003,120.425003,2.168250e+07,627.698106,135.359474,inf
